# RACE RC & Quiz Generation - Final GPU Training (Rubric Compliant)
This notebook mirrors `model_a_train.py` for full-dataset training on Google Colab T4 GPU. 
It includes scaling, supervised models (LR, SVM, NB), unsupervised models, and the ensemble.

In [ ]:
# 1. Setup & Installation
!pip install cuml-cu12 cudf-cu12 --extra-index-url https://pypi.nvidia.com
!pip install gensim joblib pandas numpy scikit-learn

import os, joblib, time
import numpy as np
import pandas as pd
import cupy as cp
from google.colab import drive
drive.mount('/content/drive')

# Paths
PROJECT_DIR = '/content/drive/MyDrive/race_rc_project'
DATA_DIR    = f'{PROJECT_DIR}/data/processed'
MODEL_DIR_A = f'{PROJECT_DIR}/models/model_a/traditional'
os.makedirs(MODEL_DIR_A, exist_ok=True)

# Load Data
print("Loading features...")
train_data = joblib.load(f'{DATA_DIR}/train_features.pkl')
val_data   = joblib.load(f'{DATA_DIR}/val_features.pkl')

X_train, y_train = train_data['X'], train_data['y']
X_val, y_val     = val_data['X'], val_data['y']
print(f"Loaded {len(X_train)} training rows.")

## 2. Feature Scaling (Rubric Requirement 1)
Consistent with `model_a_train.py`, we fit a scaler to the training data.

In [ ]:
from sklearn.preprocessing import StandardScaler

print("Fitting StandardScaler...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

joblib.dump(scaler, f'{MODEL_DIR_A}/scaler.pkl')
print("Scaler saved to models/model_a/traditional/scaler.pkl")

## 3. Supervised Models (Rubric Requirement 2)
Training Logistic Regression, LinearSVC (GPU), and BernoulliNB.

In [ ]:
from cuml.svm import LinearSVC as cuSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import BernoulliNB
from sklearn.calibration import CalibratedClassifierCV

# 3a. Logistic Regression
print("Training Logistic Regression...")
lr = LogisticRegression(class_weight='balanced', max_iter=1000, n_jobs=-1)
lr.fit(X_train_scaled, y_train)
joblib.dump(lr, f'{MODEL_DIR_A}/lr.pkl')

# 3b. SVM (GPU Accelerated)
print("Training LinearSVC on GPU...")
X_gpu = cp.asarray(X_train_scaled.astype('float32'))
y_gpu = cp.asarray(y_train.astype('float32'))
svm_raw = cuSVC(class_weight='balanced', C=0.5, max_iter=2000)
svm_raw.fit(X_gpu, y_gpu)
joblib.dump(svm_raw, f'{MODEL_DIR_A}/svm.pkl')

# 3c. Naive Bayes (on binary features)
print("Training Naive Bayes...")
X_tr_bin = (X_train_scaled > 0).astype(np.float32)
nb = BernoulliNB(alpha=1.0)
nb.fit(X_tr_bin, y_train)
joblib.dump(nb, f'{MODEL_DIR_A}/nb.pkl')

## 4. Ensemble Achievement (Rubric Requirement 4)
Soft-voting ensemble of the calibrated classifiers.

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV

print("Building Ensemble (Soft Voting)...")
# Calibrate SVM for probabilities
svm_cal = CalibratedClassifierCV(svm_raw, cv='prefit')
svm_cal.fit(X_val_scaled, y_val)

ensemble = VotingClassifier(
    estimators=[
        ('lr', lr), 
        ('svm', svm_cal), 
        ('nb', nb)
    ], 
    voting='soft'
)
ensemble.fit(X_train_scaled, y_train) # Re-fit or fit on small subset if time-limited
joblib.dump(ensemble, f'{MODEL_DIR_A}/ensemble.pkl')
print("Ensemble saved.")

## 5. Unsupervised Approach (Rubric Requirement 3)
K-Means clustering and Silhouette scoring.

In [ ]:
from cuml.cluster import KMeans as cuKMeans
print("Running K-Means on GPU...")
km = cuKMeans(n_clusters=4, random_state=42)
km.fit(X_gpu)
joblib.dump(km, f'{MODEL_DIR_A}/kmeans.pkl')
print("Unsupervised training complete.")